# Load merged gameweek data

This notebook loads `merged_gw_all_seasons_slim.csv` from the project folder.

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from sqlalchemy import create_engine
from sklearn.inspection import permutation_importance
# XGBoost regression (run after scaling; uses same X_train / X_test as linear / poly)
from sklearn.base import clone
from xgboost import XGBRegressor
# pd.set_option("display.max_columns", None)

In [9]:
def load_data(path: Path = Path("fpl_final.csv")) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    # df_original=df.copy()
    return df

In [10]:
# Cleaning the dataset
# Remove rows where position is defender or goalkeeper (keep MID, FWD, etc.)
# if "position" in df.columns:
#     df = df[~df["position"].isin(["DEF", "GK"])].copy()


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    initial_cols_to_drop = [
        'season', "penalties_conceded", "big_chances_created", "big_chances_missed", "name", 'bonus',
        'clean_sheets', 'creativity', 'goals_conceded', 'goals_scored', 'ict_index', 'influence',
        'kickoff_time', 'kickoff_time_formatted', 'minutes', 'GW', 'assists', 'own_goals', 'red_cards',
        'round', 'selected', 'team_a_score', 'team_h_score', 'threat', 'transfers_balance', 'transfers_in',
        'penalties_missed', 'penalties_saved', 'position', 'saves', 'transfers_out', 'value', 'yellow_cards',
        "big_chances_created_prev_3_mean", "big_chances_created_prev_all_mean", "big_chances_missed_prev_3_mean",
        "big_chances_missed_prev_all_mean", "assists_prev_5_mean", "goals_conceded_prev_5_mean",
        'goals_scored_prev_5_mean', "total_points_prev_5_same_opponent_mean", "penalties_saved_prev_3_mean",
        "penalties_saved_prev_all_mean", "team_a_score_prev_3_mean", "team_a_score_prev_all_mean",
        "team_h_score_prev_3_mean", "team_h_score_prev_all_mean", "opponent_strength_overall_away",
        "opponent_strength_overall_home", "self_team_strength_overall_away", "self_team_strength_overall_home",
        'goals_scored_prev_all_mean', "clean_sheets_prev_all_mean", "threat_prev_all_mean",
        "creativity_prev_all_mean", 'goals_conceded_prev_all_mean', 'saves_prev_3_mean', 'assists_prev_all_mean',
        'saves_prev_all_mean', 'team', 'opponent_team_name', 'total_points_prev_3_same_opponent_mean',
        'goals_scored_prev_3_mean', 'goals_conceded_prev_3_mean', 'clean_sheets_prev_3_mean',
        'assists_prev_3_mean', "player_image_url"
    ]
    cleaned_df = df.drop(columns=initial_cols_to_drop, errors="ignore")
    print("Total missing values:", cleaned_df.isna().sum().sum())
    print(cleaned_df.shape)
    cleaned_df.columns[cleaned_df.isna().any()].tolist()
    
    return cleaned_df


In [11]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "total_points_prev_5_mean" in df.columns:
        form_source = df["total_points_prev_5_mean"]
    elif {"total_points_prev_3_mean", "total_points_prev_all_mean"}.issubset(df.columns):
        form_source = 0.7 * df["total_points_prev_3_mean"] + 0.3 * df["total_points_prev_all_mean"]
    elif "total_points_prev_3_mean" in df.columns:
        form_source = df["total_points_prev_3_mean"]
    elif "total_points_prev_all_mean" in df.columns:
        form_source = df["total_points_prev_all_mean"]
    else:
        form_source = 0

    df["form"] = pd.to_numeric(form_source, errors="coerce").fillna(0.0)
    return df

In [12]:
def save_processed_data(df):
    # Create database file (will be created automatically)
    engine = create_engine("sqlite:///fpl_data.db")

    # Save dataframe to table
    df.to_sql("fpl_processed", engine, if_exists="replace", index=False)

    print("Data saved to SQLite successfully!")

In [13]:
def pipeline() -> pd.DataFrame:
    df = load_data()
    df = clean_data(df)
    df = feature_engineering(df)
    save_processed_data(df)
    return df

df_original=load_data()
df = pipeline()
print(df[["form"]].head())

Total missing values: 0
(246496, 18)
Data saved to SQLite successfully!
       form
0  1.257477
1  1.257477
2  1.257477
3  1.257477
4  1.257477


In [14]:
target_col = "total_points"
# cat_cols = ["opponent_team_name", "team"]

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

ord_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
# X_train[cat_cols] = ord_enc.fit_transform(X_train[cat_cols])
# X_test[cat_cols] = ord_enc.transform(X_test[cat_cols])

# X_train["was_home"] = X_train["was_home"].astype("int8")
# X_test["was_home"] = X_test["was_home"].astype("int8")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (197196, 18), X_test: (49300, 18)
y_train: (197196,), y_test: (49300,)


In [15]:


scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)
print(X_train.shape, X_test.shape)

(197196, 18) (49300, 18)


In [16]:


xgb_model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=44,
    n_jobs=-1,
    tree_method="hist",
)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

r2_train = float(r2_score(y_train, y_train_pred))
r2_test = float(r2_score(y_test, y_test_pred))
rmse_test = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
mae_test = float(mean_absolute_error(y_test, y_test_pred))

mape = float(
    np.mean(
        np.abs(y_test.values - y_test_pred)
        / np.maximum(np.abs(y_test.values), 0.5)
    )
    * 100
)
within_one = float(np.mean(np.abs(y_test.values - y_test_pred) <= 1.0) * 100)

print("XGBoost — metrics")
print(f"  R2 train: {r2_train * 100:.2f}%  |  R2 test: {r2_test * 100:.2f}%")
print(f"  RMSE test: {rmse_test:.4f}  |  MAE test: {mae_test:.4f}")
print(f"  MAPE (approx): {mape:.2f}%  |  |error|<=1 pt: {within_one:.2f}%")

# Bootstrap refits (B smaller than linear/poly — each fit is heavier)
rng = np.random.default_rng(45)
B = 30
n_train, n_test = len(X_train), len(X_test)
preds_boot = np.empty((B, n_test))
for b in range(B):
    idx = rng.integers(0, n_train, size=n_train)
    m = clone(xgb_model).fit(X_train.iloc[idx], y_train.iloc[idx])
    preds_boot[b] = m.predict(X_test)

avg_var = float(np.mean(preds_boot.var(axis=0, ddof=1)))
bias_mean = float(np.mean(y_test_pred - y_test.values))
print(f"  Bootstrap B={B}: mean residual {bias_mean:.4f}, avg pred var {avg_var:.6f}")

# fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# axes[0].scatter(y_test, y_test_pred, alpha=0.12, s=10, c="C2")
# mx = float(max(np.max(y_test), np.max(y_test_pred)))
# axes[0].plot([0, mx], [0, mx], "r--", lw=1.5, label="ideal")
# axes[0].set_xlabel("Actual total_points")
# axes[0].set_ylabel("Predicted")
# axes[0].set_title("XGBoost — test")
# axes[0].legend()
# axes[1].hist(y_test.values - y_test_pred, bins=45, edgecolor="black", alpha=0.75, color="C2")
# axes[1].axvline(0.0, color="red", linestyle="--", lw=1.5)
# axes[1].set_xlabel("Residual")
# axes[1].set_title("XGBoost — test residuals")
# plt.tight_layout()
# plt.show()

# imp = xgb_model.feature_importances_
# order = np.argsort(imp)[::-1][: min(15, len(imp))]
# plt.figure(figsize=(8, 4))
# plt.barh(np.array(X_train.columns)[order][::-1], imp[order][::-1], color="C2", alpha=0.85)
# plt.xlabel("Gain-based importance")
# plt.title("XGBoost — top feature importances")
# plt.tight_layout()
# plt.show()


XGBoost — metrics
  R2 train: 37.85%  |  R2 test: 28.78%
  RMSE test: 2.0618  |  MAE test: 1.1145
  MAPE (approx): 91.75%  |  |error|<=1 pt: 66.04%
  Bootstrap B=30: mean residual 0.0012, avg pred var 0.050296


In [17]:
def feature_importance():

    result = permutation_importance(
    xgb_model, X_test, y_test, n_repeats=10, random_state=42)

    perm_imp = pd.Series(result.importances_mean, index=X.columns)
    perm_imp = perm_imp.sort_values(ascending=False)

    # print(perm_imp[:])

In [28]:
import json
import joblib

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

xgb_model.save_model(artifacts_dir / "xgb_model.json")
joblib.dump(scaler, artifacts_dir / "scaler.joblib")
(artifacts_dir / "metadata.json").write_text(
    json.dumps({"feature_columns": X.columns.tolist()}, indent=2)
)

print(f"Saved model artifacts to {artifacts_dir.resolve()}")


Saved model artifacts to /Users/shimantoa./Documents/Data Cleaning/updated_fpl_project/artifacts


In [34]:
from service import FPLPredictionService

svc = FPLPredictionService()
result = svc.predict_best_xi()

best_xi = pd.DataFrame(result["players"])

print(
    f"Best predicted lineup for season {result['season']}, GW {result['gameweek']}"
)
print(f"Formation: {result['formation']}")
print(f"Total predicted points: {result['total_predicted_points']:.2f}")
display(best_xi.reset_index(drop=True))


Best predicted lineup for season 2026, GW 29
Formation: {'GK': 2, 'DEF': 5, 'MID': 5, 'FWD': 3}
Total predicted points: 74.88


,name,position,team,opponent_team_name,predicted_points
0,Bernd Leno,GK,Fulham,West Ham,4.67
1,Robin Roefs,GK,Sunderland,Leeds,4.41
2,James Hill,DEF,Bournemouth,Brentford,5.13
3,Marcos Senesi Barón,DEF,Bournemouth,Brentford,4.73
4,Jayden Bogle,DEF,Leeds,Sunderland,4.05
5,Lewis Hall,DEF,Newcastle,Man Utd,3.84
6,Omar Alderete,DEF,Sunderland,Leeds,3.75
7,Kiernan Dewsbury-Hall,MID,Everton,Burnley,5.72
8,Bruno Borges Fernandes,MID,Man Utd,Newcastle,5.57
9,Bukayo Saka,MID,Arsenal,Brighton,5.56
